# A first tour of sonore

[sonore](https://github.com/choyun1/sonore) makes, manipulates and analyzes sounds the way hearing
scientists think about them. This notebook is a short tour you can run in the browser: on
[Google Colab](https://colab.research.google.com/github/choyun1/sonore/blob/main/docs/notebooks/start.ipynb)
it needs nothing but a Google account. Run the cells from top to bottom (Runtime > Run all). Every
sound comes with an audio player and four views of it: its waveform, spectrum, spectrogram and
modulation spectrum. Headphones help, and are needed for the binaural example.

The [listening gallery](https://choyun1.github.io/sonore/gallery/) goes much further, with the code
for each example beside the sound it makes.

In [ ]:
%pip install --quiet sonore

In [ ]:
import matplotlib.pyplot as plt

import sonore as so
from sonore import dB

fs = 44100
print("sonore", so.__version__)


def show(*sounds, fmax=None):
    """Plays each sound, with its overview below: waveform, spectrum, spectrogram and modulation
    spectrum."""
    for sound in sounds:
        display(sound)
        so.overview(sound, figsize=(8.4, 5.6), fmax=fmax)
        plt.show()

## Making sounds

Every generator returns a `Sound` with an RMS of 1, and anything stochastic takes `rng=` so it
can be reproduced exactly. A player plays a sound at its true level and can't go above full scale,
so the sounds here are set 20 dB down, to an RMS of 0.1. `show`, defined in the cell above, plays each sound it is given and
draws its overview below the player with `so.overview`: the waveform, the spectrum, the
spectrogram and the modulation spectrum.

In [ ]:
tone = so.pure_tone(0.5, fs, 1000).ramp(10e-3) - 20 * dB
show(tone)

In [ ]:
complex_ = so.harmonic_complex(0.5, fs, f0=200, harmonics=range(1, 21), phases="schroeder+")
complex_ = complex_.ramp(10e-3) - 20 * dB
noise = so.gaussian_noise(0.5, fs, band=(100, 8000), tilt=-3, rng=0)  # pink, band-limited
noise = noise.ramp(10e-3) - 20 * dB
show(complex_, noise)

## Levels and time

Levels are written as levels: adding decibels changes a sound's level, and adding two sounds mixes
them. Times are in seconds, so `snd[0.1:0.4]` is the stretch from 100 to 400 ms.

In [ ]:
tone_in_noise = tone + (noise + 5 * dB)  # the tone at -5 dB SNR
show(tone_in_noise, tone_in_noise[0.1:0.4])

## A spoken sentence

sonore does not ship recordings, so this cell downloads one from the sonore repository on GitHub:
"Providence had delivered him through the maelstrom", from the CMU ARCTIC corpus (Kominek & Black,
2004), whose licence allows this use. Its sources and licence are in
[docs/speech/SOURCES.md](https://github.com/choyun1/sonore/blob/main/docs/speech/SOURCES.md).
`so.load` reads any audio file that [soundfile](https://python-soundfile.readthedocs.io) can.

In [ ]:
import os
import urllib.request

SENTENCE = "bdl_arctic_a0131.flac"
if not os.path.exists(SENTENCE):
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/choyun1/sonore/main/docs/speech/" + SENTENCE, SENTENCE
    )
sentence = so.load(SENTENCE)
show(sentence, fmax=5000)

## Two spectrograms

The window length decides what a spectrogram shows. A 5 ms window is shorter than one period of
this voice, so each glottal pulse stands out as a vertical stripe (a *wideband* spectrogram). A
33 ms window holds several periods, so the harmonics stand out as horizontal lines instead (a
*narrowband* one). The [Seeing speech](https://choyun1.github.io/sonore/gallery/speech.html) page
explains why, and what else a time-frequency picture can be.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8.4, 2.8), layout="constrained")
for ax, win in zip(axes, [5e-3, 33e-3], strict=True):
    so.STFT(sentence, win_dur=win).plot(ax=ax, db_range=70, fmax=5000, colorbar=False)
    ax.set_title(f"{win * 1e3:.0f} ms window")

## The cepstrum

The cepstrum is the Fourier transform of the log spectrum. The harmonics ripple the log spectrum
once every F0 hertz, so the voice's pitch shows up as a peak at one period. Doing this for every
time window gives a *cepstrogram*, and picking its peak gives a pitch track. The
[Cepstral analysis](https://choyun1.github.io/sonore/gallery/cepstrum.html) page uses it to hear
a voice's pitch and timbre separately.

In [ ]:
cep = so.Cepstrum(so.STFT(sentence, win_dur=0.040, hop_dur=0.005))
t, f0, _ = cep.f0()
voiced = f0[0] > 0

fig, (ax0, ax1) = plt.subplots(2, 1, figsize=(8.4, 4.6), sharex=True, layout="constrained")
cep.plot(ax=ax0, colorbar=False)
ax1.plot(t[voiced], f0[0, voiced], ".", ms=3)
ax1.set(xlabel="Time [s]", ylabel="F0 [Hz]", title="Pitch from the cepstral peak", ylim=(50, 250))
ax1.grid(ls=":")

## Changing duration and pitch

A phase vocoder changes duration without changing pitch, and pitch without changing duration. The
[Phase vocoder](https://choyun1.github.io/sonore/gallery/pv.html) page shows how it works.

In [ ]:
slower = so.time_stretch(sentence, 1.5)  # same pitch, half again as long
lower = so.pitch_shift(sentence, -5)  # same duration, five semitones down
show(slower, lower, fmax=5000)

## A moving ripple and its modulation spectrum

A ripple is a sinusoidal pattern drawn on the time-frequency plane. This one drifts down at 4 Hz
with one cycle per octave, and you hear a sweep four times a second. The modulation spectrum, the
two-dimensional Fourier transform of the sound's envelopes, measures it back as a single peak.
More on the [Spectrotemporal ripples](https://choyun1.github.io/sonore/gallery/ripples.html) page.

In [ ]:
ripple = so.ripple_sound(so.Ripple(4, 1), 2, fs, rng=0).ramp(10e-3) - 20 * dB
show(ripple)
so.ModulationSpectrum.octave(ripple, f_lo=250, f_hi=8000).plot(wt_max=20, wf_max=4);

## Binaural cues

With headphones, a 500 µs interaural time difference and a 6 dB level difference move noise to
the right. `so.interaural_cues` measures both back in short windows. More on the
[Binaural cues](https://choyun1.github.io/sonore/gallery/binaural.html) page.

In [ ]:
lateral = so.apply_itd_ild(noise, itd=500e-6, ild=6)
show(lateral)
so.interaural_cues(lateral, win_dur=20e-3).plot();

## Where next

- The [listening gallery](https://choyun1.github.io/sonore/gallery/): textures, reverberation,
  moving talkers, vocoders, iterated rippled noise, classic stimuli, and more, each beside its
  code.
- The [README](https://github.com/choyun1/sonore#readme): conventions, and every module and what
  it holds.
- On your own machine: `pip install "sonore[notebook]"`.